# Dataset Split by Classifier Correctness and Cosine Similarity

This notebook accepts either `nonsarcastic_to_sarcastic_out.json` or `sarcastic_to_nonsarcastic_out.json` at runtime, flips the original `is_sarcastic` label to get the target label for `reversed_headline`, runs the same sarcasm classifier used in `data_pair_evaluation.ipynb`, computes the same cosine similarity metric used in `cosine_similarity_distribution.ipynb`, and exports three JSONL files:

- rows where `reversed_headline` is classified incorrectly
- rows classified correctly and with cosine similarity `>= 0.4`
- rows classified correctly and with cosine similarity `< 0.4`

The output files preserve only the original dataset fields.


In [ ]:
# Pull code, install dependencies

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LINCHENYU2030S/CS4248_group_project.git"
REPO_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_EVAL_DIR = LOCAL_PROJECT_ROOT / "evaluation"
LOCAL_EVAL_DIR = Path("/content/evaluation")

current_dir = Path.cwd().resolve()
if current_dir.name == "evaluation":
    project_eval_dir = current_dir
elif current_dir.name == "notebooks" and (current_dir.parent / "evaluation").exists():
    project_eval_dir = current_dir.parent / "evaluation"
elif (current_dir / "evaluation").exists():
    project_eval_dir = current_dir / "evaluation"
elif LOCAL_PROJECT_EVAL_DIR.exists():
    project_eval_dir = LOCAL_PROJECT_EVAL_DIR
elif LOCAL_EVAL_DIR.exists():
    project_eval_dir = LOCAL_EVAL_DIR
elif "google.colab" in sys.modules:
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    project_eval_dir = REPO_ROOT / "evaluation"
else:
    raise RuntimeError(
        f"Could not locate the repo evaluation directory from: {current_dir}"
    )

os.chdir(project_eval_dir)
PROJECT_EVAL_DIR = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_EVAL_DIR.parent
if PROJECT_EVAL_DIR.name != "evaluation":
    raise RuntimeError(f"Expected to run inside the evaluation directory, got: {PROJECT_EVAL_DIR}")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
for path in [PROJECT_ROOT, PROJECT_EVAL_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "sentence-transformers>=2.6.0",
        "transformers>=4.39.0",
        "accelerate>=0.28.0",
        "scikit-learn>=1.4.0",
        "pandas>=2.0.0",
        "matplotlib>=3.7.0",
        "ipywidgets>=8.1.0",
        "tqdm>=4.66.0",
    ],
    check=True,
)

print(f"Working directory: {PROJECT_EVAL_DIR}")
print(f"Project root: {PROJECT_ROOT}")


In [1]:
# Upload one supported dataset at run time

import shutil

ALLOWED_DATASET_FILENAMES = {
    "nonsarcastic_to_sarcastic_out.json",
    "sarcastic_to_nonsarcastic_out.json",
}
runtime_paths = {
    filename: PROJECT_EVAL_DIR / filename
    for filename in ALLOWED_DATASET_FILENAMES
}

existing_candidates = [path for path in runtime_paths.values() if path.exists()]
if len(existing_candidates) == 1:
    DATASET_PATH = existing_candidates[0]
    DATASET_FILENAME = DATASET_PATH.name
    print(f"Using dataset already available at: {DATASET_PATH}")
elif len(existing_candidates) > 1:
    raise RuntimeError(
        "Multiple supported datasets are already present in the runtime evaluation directory. "
        "Please remove one of them or restart the runtime so the notebook knows which one to use."
    )
elif "google.colab" in sys.modules:
    from google.colab import files

    print("Upload exactly one of these files:")
    print(sorted(ALLOWED_DATASET_FILENAMES))
    uploaded = files.upload()

    uploaded_candidates = [name for name in uploaded if name in ALLOWED_DATASET_FILENAMES]
    if len(uploaded_candidates) != 1:
        raise FileNotFoundError(
            "Expected exactly one uploaded dataset with a supported filename. "
            f"Uploaded files: {list(uploaded)}"
        )

    DATASET_FILENAME = uploaded_candidates[0]
    DATASET_PATH = runtime_paths[DATASET_FILENAME]
    upload_candidates = [Path("/") / DATASET_FILENAME, Path.cwd() / DATASET_FILENAME]
    upload_source = next((path for path in upload_candidates if path.exists()), None)
    if upload_source is None:
        raise FileNotFoundError(
            f"Uploaded file not found in expected locations: {upload_candidates}"
        )

    shutil.move(str(upload_source), str(DATASET_PATH))
    print(f"Dataset saved to: {DATASET_PATH}")
else:
    raise FileNotFoundError(
        "No supported dataset found in the runtime directory. Place one of the allowed files there before continuing."
    )


NameError: name 'PROJECT_EVAL_DIR' is not defined

In [ ]:
# Imports, configuration, and device setup

import json
import random
import string

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, f1_score
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed

from evaluation.text_similarity import (
    DEFAULT_MODEL_NAME as DEFAULT_SIMILARITY_MODEL_NAME,
    batch_cosine_similarity,
    load_embedding_model,
)

pd.set_option("display.max_colwidth", 120)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE != "cuda":
    print("Warning: GPU is not enabled. This notebook is intended for Colab T4, so CPU will be slower.")

CLASSIFIER_MODEL_NAME = "helinivan/english-sarcasm-detector"
SIMILARITY_MODEL_NAME = DEFAULT_SIMILARITY_MODEL_NAME
HEADLINE_COLUMN = "headline"
REVERSED_HEADLINE_COLUMN = "reversed_headline"
ORIGINAL_LABEL_COLUMN = "is_sarcastic"
FLIPPED_LABEL_COLUMN = "reversed_label"
COSINE_COLUMN = "cosine_similarity"
CLASSIFIER_BATCH_SIZE = 64
SIMILARITY_BATCH_SIZE = 256
MAX_LENGTH = 256
HIGH_SIMILARITY_THRESHOLD = 0.4
LOW_SIMILARITY_THRESHOLD = 0.4

OUTPUT_STEM = Path(DATASET_FILENAME).stem
INCORRECT_OUTPUT_PATH = PROJECT_EVAL_DIR / f"{OUTPUT_STEM}_classifier_incorrect.json"
CORRECT_HIGH_SIM_OUTPUT_PATH = PROJECT_EVAL_DIR / f"{OUTPUT_STEM}_classifier_correct_similarity_ge_0p4.json"
CORRECT_LOW_SIM_OUTPUT_PATH = PROJECT_EVAL_DIR / f"{OUTPUT_STEM}_classifier_correct_similarity_lt_0p4.json"

print(f"Dataset path: {DATASET_PATH}")
print(f"Classifier model: {CLASSIFIER_MODEL_NAME}")
print(f"Similarity model: {SIMILARITY_MODEL_NAME}")


In [ ]:
# Load dataset and flip the label for reversed_headline

try:
    df = pd.read_json(DATASET_PATH, lines=True)
except ValueError:
    df = pd.read_json(DATASET_PATH)

required_columns = {HEADLINE_COLUMN, REVERSED_HEADLINE_COLUMN, ORIGINAL_LABEL_COLUMN}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

df = df.dropna(subset=[HEADLINE_COLUMN, REVERSED_HEADLINE_COLUMN, ORIGINAL_LABEL_COLUMN]).copy()
df[HEADLINE_COLUMN] = df[HEADLINE_COLUMN].astype(str).str.strip()
df[REVERSED_HEADLINE_COLUMN] = df[REVERSED_HEADLINE_COLUMN].astype(str).str.strip()
df[ORIGINAL_LABEL_COLUMN] = df[ORIGINAL_LABEL_COLUMN].astype(int)
df = df[(df[HEADLINE_COLUMN] != "") & (df[REVERSED_HEADLINE_COLUMN] != "")].copy()

invalid_labels = set(df[ORIGINAL_LABEL_COLUMN].unique()) - {0, 1}
if invalid_labels:
    raise ValueError(f"Original labels must contain only 0/1 values. Found: {sorted(invalid_labels)}")

ORIGINAL_COLUMNS = df.columns.tolist()
df[FLIPPED_LABEL_COLUMN] = 1 - df[ORIGINAL_LABEL_COLUMN]

display(df[ORIGINAL_COLUMNS + [FLIPPED_LABEL_COLUMN]].head())
print(f"Rows loaded: {len(df):,}")


In [ ]:
# Run the same sarcasm classifier used in data_pair_evaluation.ipynb

def preprocess_for_classifier(text: str) -> str:
    return text.lower().translate(str.maketrans("", "", string.punctuation)).strip()

tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)
classifier = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_MODEL_NAME).to(DEVICE)
classifier.eval()

def predict_sarcasm(texts, batch_size=CLASSIFIER_BATCH_SIZE, max_length=MAX_LENGTH):
    predictions = []
    confidences = []
    sarcastic_probabilities = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Classifier inference"):
        batch_texts = [preprocess_for_classifier(text) for text in texts[start:start + batch_size]]
        tokenized = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)

        with torch.inference_mode():
            logits = classifier(**tokenized).logits
            probabilities = torch.softmax(logits, dim=-1)

        batch_confidences, batch_predictions = probabilities.max(dim=-1)
        predictions.extend(batch_predictions.cpu().tolist())
        confidences.extend(batch_confidences.cpu().tolist())
        sarcastic_probabilities.extend(probabilities[:, 1].cpu().tolist())

    return predictions, confidences, sarcastic_probabilities

predictions, confidences, sarcastic_probabilities = predict_sarcasm(df[REVERSED_HEADLINE_COLUMN].tolist())
df["predicted_label"] = predictions
df["prediction_confidence"] = confidences
df["sarcastic_probability"] = sarcastic_probabilities

accuracy = accuracy_score(df[FLIPPED_LABEL_COLUMN], df["predicted_label"])
macro_f1 = f1_score(df[FLIPPED_LABEL_COLUMN], df["predicted_label"], average="macro")
print(f"Classifier accuracy: {accuracy:.4f}")
print(f"Classifier macro-F1: {macro_f1:.4f}")
display(
    pd.DataFrame(
        classification_report(
            df[FLIPPED_LABEL_COLUMN],
            df["predicted_label"],
            target_names=["not_sarcastic", "sarcastic"],
            output_dict=True,
            zero_division=0,
        )
    ).transpose()
)


In [ ]:
# Compute the same cosine similarity metric used in cosine_similarity_distribution.ipynb

print(f"Loading similarity model: {SIMILARITY_MODEL_NAME}")
_ = load_embedding_model(SIMILARITY_MODEL_NAME)

headlines = df[HEADLINE_COLUMN].tolist()
reversed_headlines = df[REVERSED_HEADLINE_COLUMN].tolist()
all_similarities = []

for start in tqdm(range(0, len(df), SIMILARITY_BATCH_SIZE), desc="Cosine similarity"):
    end = start + SIMILARITY_BATCH_SIZE
    batch_scores = batch_cosine_similarity(
        headlines[start:end],
        reversed_headlines[start:end],
        model_name=SIMILARITY_MODEL_NAME,
    )
    all_similarities.extend(batch_scores)

df[COSINE_COLUMN] = all_similarities
display(df[[HEADLINE_COLUMN, REVERSED_HEADLINE_COLUMN, COSINE_COLUMN]].head())


In [ ]:
# Split into incorrect, correct-high-similarity, and correct-low-similarity groups

incorrect_df = df[df["predicted_label"] != df[FLIPPED_LABEL_COLUMN]].copy()
correct_df = df[df["predicted_label"] == df[FLIPPED_LABEL_COLUMN]].copy()
correct_high_similarity_df = correct_df[correct_df[COSINE_COLUMN] >= HIGH_SIMILARITY_THRESHOLD].copy()
correct_low_similarity_df = correct_df[correct_df[COSINE_COLUMN] < LOW_SIMILARITY_THRESHOLD].copy()

summary = pd.DataFrame(
    [
        {
            "total_rows": len(df),
            "classifier_incorrect": len(incorrect_df),
            "classifier_correct": len(correct_df),
            "classifier_correct_similarity_ge_0p4": len(correct_high_similarity_df),
            "classifier_correct_similarity_lt_0p4": len(correct_low_similarity_df),
        }
    ]
)
display(summary)

display(incorrect_df[ORIGINAL_COLUMNS + [FLIPPED_LABEL_COLUMN, "predicted_label", COSINE_COLUMN]].head())
display(correct_high_similarity_df[ORIGINAL_COLUMNS + [COSINE_COLUMN]].head())
display(correct_low_similarity_df[ORIGINAL_COLUMNS + [COSINE_COLUMN]].head())


In [ ]:
# Write the three requested JSONL output files, preserving only the original dataset fields

def write_jsonl(dataframe: pd.DataFrame, output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as fout:
        for record in dataframe[ORIGINAL_COLUMNS].to_dict(orient="records"):
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")

write_jsonl(incorrect_df, INCORRECT_OUTPUT_PATH)
write_jsonl(correct_high_similarity_df, CORRECT_HIGH_SIM_OUTPUT_PATH)
write_jsonl(correct_low_similarity_df, CORRECT_LOW_SIM_OUTPUT_PATH)

print(f"Saved incorrect set to: {INCORRECT_OUTPUT_PATH}")
print(f"Saved correct high-similarity set to: {CORRECT_HIGH_SIM_OUTPUT_PATH}")
print(f"Saved correct low-similarity set to: {CORRECT_LOW_SIM_OUTPUT_PATH}")
